In [ ]:
!pip install catboost
!pip install lime

In [ ]:
# =============================================================================
# TASK 1: DELIVERY DELAY CLASSIFICATION
# E-Commerce Logistics — Explainable ML Framework
# Author  : Rakesh S Gautham (Student ID: 1196243)
# Thesis  : Explainable Prediction of Delivery Timeliness, Delay Severity and
#           Operational Risk in E-Commerce Logistics using ML & SHAP
# Supervisor: Ekta Upadhyay | LJMU, UK | March 2026
# PIPELINE OVERVIEW
# -----------------
# 1. Data Loading and Initial Dataset Preparation
# 2. Exploratory Data Analysis (EDA) Summary
# 3. Target Variable Construction
# 4. Feature Engineering
# 5. Feature Selection
# 6. Preprocessing Pipeline
# 7. Train–Test Split (Stratified 80/20)
# 8. Model Definitions
# 9. Training and Evaluation
# 10. Model Comparison Table
# 11. Visualisations
# 12. Classification Report and Test Data Validation
# 13. Model Explainability using SHAP and LIME
# 14. Final Summary
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from time import time

# Preprocessing & Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from google.colab import drive
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.metrics import roc_curve
# Imbalance handling
from sklearn.utils.class_weight import compute_class_weight

# Hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# SHAP (optional - comment out if not installed)
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("[INFO] SHAP not installed. Skipping SHAP plots.")

import lime
import lime.lime_tabular
import os
os.makedirs("outputs", exist_ok=True)

print("=" * 70)
print("  TASK 1 — DELIVERY DELAY CLASSIFICATION (ON-TIME vs LATE)")
print("=" * 70)
# ── 1. LOAD DATASET ───────────────────────────────────────────────────────────
#Reading the dataset
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
scdata_df_raw=pd.read_csv('/content/drive/MyDrive/DataCoSupplyChainDataset.csv',encoding="latin1")
scdata_df = scdata_df_raw.copy()


# ── 2. EXPLORATORY DATA ANALYSIS (BRIEF) ─────────────────────────────────────
print("\n── 2. EDA Summary ──────────────────────────────────────────────────────")
print(f"   Shape of Dataset  : {scdata_df.shape,}")
print(f"   Missing values  : {scdata_df.isnull().sum().sum():,}")
print(f"   Duplicate rows  : {scdata_df.duplicated().sum():,}")
print(f"   Delivery Status distribution:\n{scdata_df['Delivery Status'].value_counts()}")

print(f" Dropping ID & PI columns which are irrelevant, including the ones with missing values as they are not important")
drop_col_list = [
'Category Id',
'Customer Email',
'Customer Fname',
'Customer Id',
'Customer Lname',
'Customer Password',
'Department Id',
'Order Customer Id',
'Order Id',
'Order Item Cardprod Id',
'Order Item Id',
'Product Card Id',
'Product Category Id',
'Product Description',
'Product Image',
'Order Zipcode',
'Customer Zipcode',
'Product Status',
'Customer Street'
]

scdata_df = scdata_df.drop(columns=drop_col_list)

# ── 3. TARGET VARIABLE CONSTRUCTION ──────────────────────────────────────────
# Binary: 1 = Late, 0 = On-time (includes Advance shipping, Shipping on time)
print("\n── 3. Target Variable Construction ─────────────────────────────────────")

delivery_map = {
    "Late delivery"      : 1,
    "Advance shipping"   : 0,
    "Shipping on time"   : 0,
    "Shipping canceled"  : np.nan  # exclude canceled orders (ambiguous)
}
scdata_df["del_delay_class"] = scdata_df["Delivery Status"].map(delivery_map)
scdata_df = scdata_df.dropna(subset=["del_delay_class"])
scdata_df["del_delay_class"] = scdata_df["del_delay_class"].astype(int)

class_counts = scdata_df["del_delay_class"].value_counts()
print(f"   On-time (0)  : {class_counts.get(0, 0):,}")
print(f"   Late    (1)  : {class_counts.get(1, 0):,}")
print(f"   Late rate    : {class_counts.get(1,0)/len(scdata_df)*100:.1f}%")

# ── 4. FEATURE ENGINEERING ───────────────────────────────────────────────────
print("\n── 4. Feature Engineering ──────────────────────────────────────────────")

# --- 4a. Parse date columns ---
date_cols = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]
for col in date_cols:
    if col in scdata_df.columns:
        scdata_df[col] = pd.to_datetime(scdata_df[col], errors="coerce")


# Order processing time
if all(c in scdata_df.columns for c in ["order date (DateOrders)", "shipping date (DateOrders)"]):
    scdata_df["order_processing_time"] = (
        scdata_df["shipping date (DateOrders)"] - scdata_df["order date (DateOrders)"]
    ).dt.days.clip(lower=0)

# --- 4f. Temporal features ---
if "order date (DateOrders)" in scdata_df.columns:
    scdata_df["order_month"]     = scdata_df["order date (DateOrders)"].dt.month
    scdata_df["order_dayofweek"] = scdata_df["order date (DateOrders)"].dt.dayofweek



# ── 5. FEATURE SELECTION ──────────────────────────────────────────────────────
# Drop columns that are leakage risks or have no predictive value
LEAKAGE_COLS = [
    "Delivery Status", "Late_delivery_risk", "order_processing_time",
    "order date (DateOrders)", "shipping date (DateOrders)","Days for shipping (real)","Days for shipment (scheduled)"
]
scdata_df = scdata_df.drop(columns=LEAKAGE_COLS)
print(f"   Feature count after engineering: {scdata_df.shape[1]}")

#── 6. PREPROCESSING ─────────────────────────────────────────────────────"
#Highly frequent values are retained and rest are grouped under 'Others'
def top_n_categories(df, col, n=15):
    top_categories = df[col].value_counts().nlargest(n).index
    return df[col].apply(lambda x: x if x in top_categories else 'Others')

# Order City
scdata_df['Order City Grouped'] = top_n_categories(scdata_df, 'Order City', 15)

# Category Name
scdata_df['Category Name Grouped'] = top_n_categories(scdata_df, 'Category Name', 15)

# Order Region
scdata_df['Order Region Grouped'] = top_n_categories(scdata_df, 'Order Region', 15)

# Order Country
scdata_df['Order Country Grouped'] = top_n_categories(scdata_df, 'Order Country', 10)

# Product Name
scdata_df['Product Name Grouped'] = top_n_categories(scdata_df, 'Product Name', 10)

# Customer City
scdata_df['Customer City Grouped'] = top_n_categories(scdata_df, 'Customer City', 10)

# Product Name
scdata_df['Customer State Grouped'] = top_n_categories(scdata_df, 'Customer State', 15)

# Product Name
scdata_df['Order State Grouped'] = top_n_categories(scdata_df, 'Order State', 15)

#Drop original cols
drop_original_cols = [
    'Category Name',
    'Product Name',
    'Order City',
    'Order Country',
    'Order Region',
    'Customer State',
    'Customer City',
    'Order State'
]

scdata_df = scdata_df.drop(columns=drop_original_cols)

print("Original categorical columns dropped. New DataFrame shape:", scdata_df.shape)
print("New columns added:", [col for col in scdata_df.columns if 'Grouped' in col])

#Post the correlation analysis the highly collinear columns are dropped
scdata_df.corr(numeric_only=True)
scdata_df = scdata_df.drop(columns=['Sales per customer', 'Sales', 'Product Price','Order Item Profit Ratio','Benefit per order'])
print("Highly collinear features removed to reduce multicollinearity . New DataFrame shape:", scdata_df.shape)

print("\n── OUTLIER HANDLING (IQR Capping / Winsorization) ─────────────────────")

# Columns to apply IQR capping
# Exclude binary/encoded columns and target
cap_cols = [
    "Order Item Discount",
    "Order Item Product Price",
    "Order Item Total",
    "Order Profit Per Order"
]

cap_cols = [c for c in cap_cols if c in scdata_df.columns]

# Store capping bounds for documentation
capping_log = []

for col in cap_cols:
    Q1  = scdata_df[col].quantile(0.25)
    Q3  = scdata_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Count before capping
    n_lower = (scdata_df[col] < lower).sum()
    n_upper = (scdata_df[col] > upper).sum()

    # Apply capping
    scdata_df[col] = scdata_df[col].clip(lower=lower, upper=upper)

    capping_log.append({
        "Feature"      : col,
        "Lower Cap"    : round(lower, 2),
        "Upper Cap"    : round(upper, 2),
        "Capped Lower" : n_lower,
        "Capped Upper" : n_upper,
        "Total Capped" : n_lower + n_upper,
    })
    print(f"   {col:<40} lower={lower:.2f}  upper={upper:.2f}  "
          f"capped={n_lower + n_upper:,}")

capping_df = pd.DataFrame(capping_log)
capping_df.to_csv("outputs/task1_outlier_capping_log.csv", index=False)
print("\n[✓] Capping log saved → outputs/task1_outlier_capping_log.csv")

#Target variable is assigned to y
y = scdata_df['del_delay_class']
scdata_df = scdata_df.drop(columns=['del_delay_class'])

# Filter to only columns that exist
# identify numeric features for scaling and model training
num_feats = scdata_df.select_dtypes(include=['number']).columns.tolist()

# Categorical features
cat_feats = scdata_df.select_dtypes(include=['object', 'category']).columns.tolist()

# Combine
all_features = num_feats + cat_feats

print(f"\n── 5. Feature Selection ────────────────────────────────────────────────")
print(f"   Numeric features   : {len(num_feats)}")
print(f"   Categorical features: {len(cat_feats)}")
print(f"   Total features     : {len(all_features)}")

X = scdata_df[all_features].copy()

# ── 6. PREPROCESSING PIPELINE ────────────────────────────────────────────────
print("\n── 6. Preprocessing Pipeline ───────────────────────────────────────────")
# RobustScaler is used because logistics variables contain skewness and outliers
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),          # robust to outliers in logistics data
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     num_feats),
    ("cat", categorical_transformer, cat_feats),
], remainder="drop")


# ── 7. TRAIN / TEST SPLIT ─────────────────────────────────────────────────────
print("\n── 7. Train-Test Split (Stratified 80/20) ──────────────────────────────")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"   Train : {X_train.shape[0]:,} samples")
print(f"   Test  : {X_test.shape[0]:,} samples")

# Class weight for imbalance handling
classes = np.unique(y_train)
cw = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))
print(f"   Class weights : {class_weight_dict}")


# ── 8. MODEL DEFINITIONS ─────────────────────────────────────────────────────
print("\n── 8. Model Definitions ────────────────────────────────────────────────")
# Multiple machine learning models are evaluated to compare
# linear, tree-based and ensemble learning approaches
# for delivery delay classification
RANDOM_STATE = 42
N_JOBS = -1

models = {
    "Logistic Regression": Pipeline([
        ("prep",  preprocessor),
        ("clf",   LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        ))
    ]),
    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("clf",  DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=10,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        ))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("clf",  RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            max_depth=15,
            min_samples_leaf=20,
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
        ))
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("clf",  XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
            eval_metric="logloss",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
            verbosity=0,
        ))
    ]),
    "LightGBM": Pipeline([
        ("prep", preprocessor),
        ("clf",  LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=8,
            num_leaves=63,
            class_weight="balanced",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
            verbose=-1,
        ))
    ]),
    "CatBoost": Pipeline([
        ("prep", preprocessor),
        ("clf",  CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=0,
        ))
    ]),
    "SVM": Pipeline([
    ("prep", preprocessor),
    ("clf",  LinearSVC(
        class_weight="balanced",
        C=1.0,
        max_iter=2000,
        random_state=RANDOM_STATE,
    ))
]),
}

print(f"   {len(models)} models configured.")

# ── 9. TRAINING & EVALUATION ─────────────────────────────────────────────────
print("\n── 9. Training & Evaluation ────────────────────────────────────────────")

CV_FOLDS = 5
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = {}

for model_name, pipeline in models.items():
    print(f"\n   Training → {model_name} ...", end=" ", flush=True)
    t0 = time()

    # ── 9a. Cross-validation on train set (F1 macro as primary CV metric)
    cv_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=skf, scoring="f1", n_jobs=N_JOBS
    )

    # ── 9b. Fit on full train set
    pipeline.fit(X_train, y_train)
    train_time = time() - t0

    # ── 9c. Predictions
    y_pred      = pipeline.predict(X_test)
    clf = pipeline.named_steps["clf"]

    if hasattr(clf, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)[:, 1]

    elif hasattr(clf, "decision_function"):
        y_prob = pipeline.decision_function(X_test)

    else:
        raise ValueError(f"{type(clf).__name__} does not support probability scores")

    roc_auc = roc_auc_score(y_test, y_prob)

    # ── 9d. Metrics
    acc       = accuracy_score(y_test, y_pred)
    prec      = precision_score(y_test, y_pred, zero_division=0)
    rec       = recall_score(y_test, y_pred, zero_division=0)
    f1        = f1_score(y_test, y_pred, zero_division=0)
    roc_auc   = roc_auc_score(y_test, y_prob)

    results[model_name] = {
        "Accuracy"  : round(acc,     4),
        "Precision" : round(prec,    4),
        "Recall"    : round(rec,     4),
        "F1-Score"  : round(f1,      4),
        "ROC-AUC"   : round(roc_auc, 4),
        "CV F1 Mean": round(cv_scores.mean(), 4),
        "CV F1 Std" : round(cv_scores.std(),  4),
        "Train Time (s)": round(train_time, 1),
        "_pipeline" : pipeline,
        "_y_pred"   : y_pred,
        "_y_prob"   : y_prob,
    }

    print(
        f"done [{train_time:.1f}s]  "
        f"Acc={acc:.4f}  F1={f1:.4f}  ROC-AUC={roc_auc:.4f}  "
        f"CV-F1={cv_scores.mean():.4f}±{cv_scores.std():.4f}"
    )

# ── 10. COMPARISON TABLE ──────────────────────────────────────────────────────
print("\n")
print("=" * 90)
print("  MODEL COMPARISON TABLE — TASK 1: DELAY CLASSIFICATION (ON-TIME vs LATE)")
print("=" * 90)

metrics_cols = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC",
                "CV F1 Mean", "CV F1 Std", "Train Time (s)"]

comparison_df = pd.DataFrame(
    {m: {k: results[m][k] for k in metrics_cols} for m in results}
).T.reset_index().rename(columns={"index": "Model"})

# Rank by F1-Score
comparison_df = comparison_df.sort_values("F1-Score", ascending=False).reset_index(drop=True)
comparison_df.insert(0, "Rank", comparison_df.index + 1)

# Pretty print
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

print(comparison_df.to_string(index=False))
print("=" * 90)

best_model_name = comparison_df.iloc[0]["Model"]
best_f1         = comparison_df.iloc[0]["F1-Score"]
best_roc        = comparison_df.iloc[0]["ROC-AUC"]
print(f"\n   Best Model : {best_model_name}")
print(f"     F1-Score   : {best_f1:.4f}")
print(f"     ROC-AUC    : {best_roc:.4f}")
print("=" * 90)

# Save comparison CSV
comparison_df.to_csv("outputs/task1_model_comparison.csv", index=False)
print("\n[✓] Comparison table saved → outputs/task1_model_comparison.csv")

# ── 11. VISUALISATIONS ────────────────────────────────────────────────────────
print("\n── 11. Generating Visualisations ───────────────────────────────────────")

PALETTE = sns.color_palette("Set2", n_colors=len(models))
MODEL_ORDER = comparison_df["Model"].tolist()  # sorted by F1

# ── 11a. Grouped bar chart: all 5 metrics ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
bar_metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
x     = np.arange(len(MODEL_ORDER))
width = 0.15

for i, metric in enumerate(bar_metrics):
    vals = [results[m][metric] for m in MODEL_ORDER]
    bars = ax.bar(x + i * width, vals, width, label=metric, color=PALETTE[i])
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=6.5, rotation=90
        )

ax.set_xticks(x + width * 2)
ax.set_xticklabels(MODEL_ORDER, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Task 1 — Model Comparison: All Metrics", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=8)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/task1_model_comparison_bars.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved outputs/task1_model_comparison_bars.png")

# ── 11b. Heatmap of all metrics ───────────────────────────────────────────────
heatmap_df = comparison_df.set_index("Model")[bar_metrics + ["CV F1 Mean"]].astype(float)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    heatmap_df,
    annot=True, fmt=".4f",
    cmap="YlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.7},
    annot_kws={"size": 9},
)
ax.set_title("Task 1 — Model Performance Heatmap", fontsize=13, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("outputs/task1_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved outputs/task1_heatmap.png")

# ── 11c. Confusion matrices (best model) ──────────────────────────────────────
best_pipeline = results[best_model_name]["_pipeline"]
best_y_pred   = results[best_model_name]["_y_pred"]

cm = confusion_matrix(y_test, best_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["On-Time", "Late"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_model_name}", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/task1_confusion_matrix_best.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"   [✓] Saved outputs/task1_confusion_matrix_best.png")

# ── 11d. ROC curves (all models) ─────────────────────────────────────────────


fig, ax = plt.subplots(figsize=(8, 6))
for i, model_name in enumerate(MODEL_ORDER):
    fpr, tpr, _ = roc_curve(y_test, results[model_name]["_y_prob"])
    auc = results[model_name]["ROC-AUC"]
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.4f})", color=PALETTE[i], lw=1.8)

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Task 1 — ROC Curves: All Models", fontsize=13, fontweight="bold")
ax.legend(fontsize=7.5, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/task1_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved outputs/task1_roc_curves.png")

# ── 11e. CV F1 box / error bar plot ──────────────────────────────────────────
cv_means = [results[m]["CV F1 Mean"] for m in MODEL_ORDER]
cv_stds  = [results[m]["CV F1 Std"]  for m in MODEL_ORDER]

fig, ax = plt.subplots(figsize=(9, 4))
colors = [PALETTE[i] for i in range(len(MODEL_ORDER))]
bars = ax.barh(MODEL_ORDER, cv_means, xerr=cv_stds, color=colors,
               align="center", height=0.55, capsize=5, ecolor="gray")
ax.set_xlabel("CV F1-Score (mean ± std)")
ax.set_title("Task 1 — Cross-Validated F1 Score (5-Fold)", fontsize=12, fontweight="bold")
ax.set_xlim(0, 1.05)
for i, (m, s) in enumerate(zip(cv_means, cv_stds)):
    ax.text(m + s + 0.01, i, f"{m:.4f}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/task1_cv_f1.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved outputs/task1_cv_f1.png")

# ── 12. CLASSIFICATION REPORT (BEST MODEL) ────────────────────────────────────

print(f"\n── 12. Classification Report — {best_model_name} ──────────────────────")
print(classification_report(y_test, best_y_pred, target_names=["On-Time", "Late"]))

# #############################################################################
# TASK 1 — TEST DATA VALIDATION
# Binary Delay Classification (On-Time vs Late)
# Paste after Step 16 (Classification Report)
# #############################################################################

print("\n")
print("=" * 90)
print("  TEST DATA VALIDATION — TASK 1: DELIVERY DELAY CLASSIFICATION")
print(f"  Best Model : {best_model_name}")
print("=" * 90)

from scipy.stats import ttest_rel

# ── V1. Re-confirm test set predictions from best model ───────────────────────
# Explicitly re-predict on held-out test set to confirm robustness
# This is distinct from the training loop
print("\n   ── V1. Test Set Prediction Confirmation ────────────────────────────")

y_val_pred = best_pipeline.predict(X_test)
y_val_prob = results[best_model_name]["_y_prob"]

val_acc  = accuracy_score(y_test, y_val_pred)
val_prec = precision_score(y_test, y_val_pred, zero_division=0)
val_rec  = recall_score(y_test, y_val_pred, zero_division=0)
val_f1   = f1_score(y_test, y_val_pred, zero_division=0)
val_auc  = roc_auc_score(y_test, y_val_prob)

print(f"   Test Accuracy   : {val_acc:.4f}")
print(f"   Test Precision  : {val_prec:.4f}")
print(f"   Test Recall     : {val_rec:.4f}")
print(f"   Test F1-Score   : {val_f1:.4f}")
print(f"   Test ROC-AUC    : {val_auc:.4f}")

# ── V2. CV vs Test score gap — overfitting check ──────────────────────────────
print("\n   ── V2. Overfitting Check: CV vs Test Score Gap ─────────────────────")

cv_mean = results[best_model_name]["CV F1 Mean"]
cv_std  = results[best_model_name]["CV F1 Std"]
gap     = abs(cv_mean - val_f1)

print(f"   CV F1 Mean  : {cv_mean:.4f} ± {cv_std:.4f}")
print(f"   Test F1     : {val_f1:.4f}")
print(f"   Gap         : {gap:.4f}")

if gap < 0.02:
    verdict = "Excellent — model generalises well, no overfitting detected"
elif gap < 0.05:
    verdict = "Acceptable — minor gap, model is stable"
else:
    verdict = "Warning — notable gap between CV and test, review regularisation"

print(f"   Verdict     : {verdict}")

# ── V3. Paired t-test — best model vs all others ──────────────────────────────
print("\n   ── V3. Statistical Validation (Paired t-test) ──────────────────────")
print(f"   Comparing all models against best model: {best_model_name}")
print(f"\n   {'Model':<22}  {'CV F1 Mean':>10}  {'p-value':>10}  {'Significant':>13}")
print("   " + "-" * 62)

best_cv_scores = cross_val_score(
    best_pipeline, X_train, y_train,
    cv=skf, scoring="f1", n_jobs=N_JOBS
)

for model_name in MODEL_ORDER:
    if model_name == best_model_name:
        continue
    other_cv = cross_val_score(
        results[model_name]["_pipeline"], X_train, y_train,
        cv=skf, scoring="f1", n_jobs=N_JOBS
    )
    _, p = ttest_rel(best_cv_scores, other_cv)
    sig  = "Yes (p<0.05)" if p < 0.05 else "No"
    mean = results[model_name]["CV F1 Mean"]
    print(f"   {model_name:<22}  {mean:>10.4f}  {p:>10.4f}  {sig:>13}")

# ── V4. Error analysis — False Positives and False Negatives ──────────────────
print("\n   ── V4. Error Analysis ───────────────────────────────────────────────")

y_test_arr = np.array(y_test)
tn = int(((y_val_pred == 0) & (y_test_arr == 0)).sum())
fp = int(((y_val_pred == 1) & (y_test_arr == 0)).sum())
fn = int(((y_val_pred == 0) & (y_test_arr == 1)).sum())
tp = int(((y_val_pred == 1) & (y_test_arr == 1)).sum())

print(f"   True Positives  (Late correctly flagged)       : {tp:>7,}")
print(f"   True Negatives  (On-time correctly cleared)    : {tn:>7,}")
print(f"   False Positives (On-time wrongly flagged late) : {fp:>7,}")
print(f"   False Negatives (Late orders missed)           : {fn:>7,}")
print(f"\n   False Negative Rate : {fn/(fn+tp):.4f}  "
      f"(missed late deliveries — most costly error)")
print(f"   False Positive Rate : {fp/(fp+tn):.4f}  "
      f"(false alarms — operational waste)")

# ── V5. Prediction confidence distribution ────────────────────────────────────
print("\n   ── V5. Prediction Confidence Distribution ───────────────────────────")

if y_val_prob.ndim == 2:
    prob_late = y_val_prob[:, 1]
else:
    prob_late = y_val_prob

high_conf_late   = (prob_late >= 0.80).sum()
high_conf_ontime = (prob_late <= 0.20).sum()
uncertain        = ((prob_late > 0.20) & (prob_late < 0.80)).sum()

print(f"   High confidence Late    (prob >= 0.80) : {high_conf_late:>7,}  "
      f"({100*high_conf_late/len(prob_late):.1f}%)")
print(f"   High confidence On-Time (prob <= 0.20) : {high_conf_ontime:>7,}  "
      f"({100*high_conf_ontime/len(prob_late):.1f}%)")
print(f"   Uncertain (0.20 < prob < 0.80)         : {uncertain:>7,}  "
      f"({100*uncertain/len(prob_late):.1f}%)")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(prob_late, bins=40, color="#1976D2", edgecolor="white", alpha=0.85)
ax.axvline(0.5, color="red", linestyle="--", lw=1.5, label="Decision boundary (0.5)")
ax.axvline(0.80, color="orange", linestyle="--", lw=1.2, label="High confidence Late (0.80)")
ax.axvline(0.20, color="green", linestyle="--", lw=1.2, label="High confidence On-Time (0.20)")
ax.set_xlabel("Predicted Probability of Late Delivery")
ax.set_ylabel("Number of Orders")
ax.set_title(
    f"Task 1 — Prediction Confidence Distribution\n{best_model_name}",
    fontsize=11, fontweight="bold"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/task1_validation_confidence.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n   [✓] Saved outputs/task1_validation_confidence.png")

# ── V6. Validation summary ─────────────────────────────────────────────────────
print("\n   ── V6. Validation Summary ───────────────────────────────────────────")
print(f"""
   Best Model          : {best_model_name}
   Test F1-Score       : {val_f1:.4f}
   Test ROC-AUC        : {val_auc:.4f}
   CV vs Test Gap      : {gap:.4f}  ({verdict.split(' — ')[0]})
   False Negative Rate : {fn/(fn+tp):.4f}
   False Positive Rate : {fp/(fp+tn):.4f}
   High Confidence     : {100*(high_conf_late+high_conf_ontime)/len(prob_late):.1f}% of predictions


""")

print("=" * 90)

# ── 13. SHAP EXPLAINABILITY (BEST MODEL) ─────────────────────────────────────
if SHAP_AVAILABLE:
    print("\n── 13. SHAP Explainability ─────────────────────────────────────────────")
    try:
        # Transform test data through preprocessor only
        best_clf = best_pipeline.named_steps["clf"]
        prep     = best_pipeline.named_steps["prep"]
        X_test_transformed = prep.transform(X_test)

        # Use a sample for performance
        N_SHAP = min(500, X_test_transformed.shape[0])
        rng    = np.random.default_rng(RANDOM_STATE)
        idx    = rng.choice(X_test_transformed.shape[0], N_SHAP, replace=False)
        X_shap = X_test_transformed[idx]

        # Get feature names after preprocessing
        ohe_cols = prep.named_transformers_["cat"]["ohe"].get_feature_names_out(cat_feats).tolist()
        feature_names_transformed = num_feats + ohe_cols

        if hasattr(best_clf, "predict_proba"):
            # SHAP Explainer — TreeExplainer for tree-based models, generic otherwise
            explainer = shap.TreeExplainer(best_clf) if hasattr(best_clf, "feature_importances_") \
                        else shap.Explainer(best_clf, X_shap)
            shap_values = explainer.shap_values(X_shap)

            # Handle both list-of-arrays and 3D numpy array formats
            # Tree models can return either depending on the library version
            if isinstance(shap_values, list):
                # List format: one array per class — extract class 1 (Late)
                sv = shap_values[1]
            elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
                # 3D array format: (n_samples, n_features, n_classes) — extract class 1
                sv = shap_values[:, :, 1]
            else:
                # Single 2D array (binary model) — use as-is
                sv = shap_values

            shap_df = pd.DataFrame(X_shap, columns=feature_names_transformed)

            # Summary dot plot — shows feature impact direction for Late delivery
            plt.figure(figsize=(10, 7))
            shap.summary_plot(
                sv, shap_df,
                max_display=15, show=False,
                plot_type="dot"
            )
            plt.title(f"SHAP Summary Plot — {best_model_name} (Late Delivery)", fontsize=11)
            plt.tight_layout()
            plt.savefig("outputs/task1_shap_summary.png", dpi=150, bbox_inches="tight")
            plt.close()
            print("   [✓] Saved outputs/task1_shap_summary.png")

            # Bar plot — shows mean absolute SHAP importance for Late delivery
            plt.figure(figsize=(10, 6))
            shap.summary_plot(
                sv, shap_df,
                max_display=15, show=False,
                plot_type="bar"
            )
            plt.title(f"SHAP Feature Importance Bar — {best_model_name}", fontsize=11)
            plt.tight_layout()
            plt.savefig("outputs/task1_shap_bar.png", dpi=150, bbox_inches="tight")
            plt.close()
            print("   [✓] Saved outputs/task1_shap_bar.png")

        else:
            print(f"   [!] {best_model_name} does not support predict_proba — SHAP skipped")

    except Exception as e:
        print(f"   [!] SHAP failed for {best_model_name}: {e}")
else:
    print("\n── 13. SHAP Explainability — SKIPPED (shap not installed) ─────────────")
    print("   Install with: pip install shap")


# ── 13b. LIME EXPLAINABILITY (BEST MODEL) ────────────────────────────────────
print("\n── 13b. LIME Explainability ────────────────────────────────────────────")


# ── Setup: LIME needs a numpy training matrix in the original feature space.
# We use X_train (raw, before preprocessing). The predict function wraps the
# full pipeline so LIME's perturbed rows pass through preprocessing correctly.

X_train_encoded = X_train[all_features].copy()
X_test_encoded  = X_test[all_features].copy()

# Identify which column indices are categorical (LIME needs this explicitly)
cat_indices = [all_features.index(c) for c in cat_feats if c in all_features]

# Categorical feature names for LIME (maps index → list of unique values seen in train)
categorical_names = {}
for idx_c in cat_indices:
    col_name = all_features[idx_c]
    # Build consistent category list from training data
    uniq = list(X_train_encoded[col_name].astype(str).unique())
    categorical_names[idx_c] = uniq
    # Encode both train and test as integer indices
    X_train_encoded[col_name] = X_train_encoded[col_name].astype(str).apply(
        lambda v: uniq.index(v) if v in uniq else 0
    )
    X_test_encoded[col_name] = X_test_encoded[col_name].astype(str).apply(
        lambda v: uniq.index(v) if v in uniq else 0
    )

X_train_np = X_train_encoded.values.astype(float)
X_test_np  = X_test_encoded.values.astype(float)

# Predict-proba wrapper over the full pipeline
# LinearSVC (SVM) doesn't have predict_proba natively; the pipeline uses
# decision_function. We handle both gracefully.
def lime_predict_fn(raw_rows: np.ndarray) -> np.ndarray:
    df_temp = pd.DataFrame(raw_rows, columns=all_features)
    for idx_c in cat_indices:
        col_name = all_features[idx_c]
        uniq = categorical_names[idx_c]
        df_temp[col_name] = df_temp[col_name].apply(
            lambda v: uniq[int(round(v))] if 0 <= int(round(v)) < len(uniq)
                      else uniq[0]
        )
    if hasattr(best_pipeline, "predict_proba"):
        return best_pipeline.predict_proba(df_temp)
    else:
        scores = best_pipeline.decision_function(df_temp)
        prob_pos = 1 / (1 + np.exp(-scores))
        return np.column_stack([1 - prob_pos, prob_pos])

# ── Build LimeTabularExplainer on the training data
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data        = X_train_np,
    feature_names        = all_features,
    class_names          = ["On-Time", "Late"],
    categorical_features = cat_indices,
    categorical_names    = categorical_names,
    mode                 = "classification",
    random_state         = RANDOM_STATE,
    discretize_continuous = True,   # bins continuous features for readability
)
print(f"   LIME explainer built on {X_train_np.shape[0]:,} training samples.")

# ── 13b-i. Local explanation for correctly predicted LATE deliveries (True Positives)
# Shows why the model correctly identified a late delivery
y_pred_best = results[best_model_name]["_y_pred"]
y_test_arr  = y_test.values

tp_indices = np.where((y_pred_best == 1) & (y_test_arr == 1))[0]
fp_indices = np.where((y_pred_best == 1) & (y_test_arr == 0))[0]  # false alarms

N_LOCAL = min(3, len(tp_indices))
print(f"   Explaining {N_LOCAL} true-positive (correctly predicted Late) instances...")

for k in range(N_LOCAL):
    instance_idx = tp_indices[k]
    instance_raw = X_test_np[instance_idx]

    exp = lime_explainer.explain_instance(
        data_row     = instance_raw,
        predict_fn   = lime_predict_fn,
        num_features = 15,
        num_samples  = 1000,
        labels       = (1,),        # explain class = Late (1)
    )

    # Save as HTML (interactive)
    html_path = f"outputs/task1_lime_local_tp_{k+1}.html"
    exp.save_to_file(html_path)
    print(f"   [✓] Saved {html_path}")

    # Save as PNG
    fig = exp.as_pyplot_figure(label=1)
    fig.set_size_inches(10, 6)
    fig.suptitle(
        f"LIME Local Explanation — True-Positive Late Delivery (Instance {k+1})\n"
        f"Model: {best_model_name}",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    png_path = f"outputs/task1_lime_local_tp_{k+1}.png"
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   [✓] Saved {png_path}")

# ── 13b-ii. Local explanation for a FALSE POSITIVE (false alarm)
# Shows why the model incorrectly flagged an on-time order as late
if len(fp_indices) > 0:
    instance_raw = X_test_np[fp_indices[0]]

    exp_fp = lime_explainer.explain_instance(
        data_row     = instance_raw,
        predict_fn   = lime_predict_fn,
        num_features = 15,
        num_samples  = 1000,
        labels       = (1,),
    )
    exp_fp.save_to_file("outputs/task1_lime_local_fp_1.html")
    print("   [✓] Saved outputs/task1_lime_local_fp_1.html  (False Positive)")

    fig = exp_fp.as_pyplot_figure(label=1)
    fig.set_size_inches(10, 6)
    fig.suptitle(
        f"LIME Local Explanation — False Positive (On-Time predicted as Late)\n"
        f"Model: {best_model_name}",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("outputs/task1_lime_local_fp_1.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("   [✓] Saved outputs/task1_lime_local_fp_1.png")


# ── 13b-iii. Local explanation for a FALSE NEGATIVE (missed delay)
# Shows why the model failed to catch a real late delivery — the most
# operationally costly error type in logistics
fn_indices = np.where((y_pred_best == 0) & (y_test_arr == 1))[0]

if len(fn_indices) > 0:
    instance_raw = X_test_np[fn_indices[0]]

    exp_fn = lime_explainer.explain_instance(
        data_row     = instance_raw,
        predict_fn   = lime_predict_fn,
        num_features = 15,
        num_samples  = 1000,
        labels       = (1,),
    )
    exp_fn.save_to_file("outputs/task1_lime_local_fn_1.html")
    print("   [✓] Saved outputs/task1_lime_local_fn_1.html  (False Negative)")

    fig = exp_fn.as_pyplot_figure(label=1)
    fig.set_size_inches(10, 6)
    fig.suptitle(
        f"LIME Local Explanation — False Negative (Late delivery missed by model)\n"
        f"Model: {best_model_name}",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("outputs/task1_lime_local_fn_1.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("   [✓] Saved outputs/task1_lime_local_fn_1.png")

print("\n   ── LIME Summary ──────────────────────────────────────────────────────")
print(f"   Local  : {N_LOCAL} True-Positive + 1 False-Positive + 1 False-Negative instance explanations")
print("   LIME provides instance-level narrative explanations suitable for")
print("   presenting to logistics managers without an ML background.")


# ── 14. FINAL SUMMARY ────────────────────────────────────────────────────────
print("\n")
print("=" * 90)
print("  FINAL SUMMARY — TASK 1: DELAY CLASSIFICATION")
print("=" * 90)
print(comparison_df[["Rank", "Model", "Accuracy", "Precision", "Recall",
                      "F1-Score", "ROC-AUC", "CV F1 Mean", "CV F1 Std",
                      "Train Time (s)"]].to_string(index=False))
print("=" * 90)
print(f"\n  Best Performing Model  : {best_model_name}")
print(f"  F1-Score               : {best_f1:.4f}")
print(f"  ROC-AUC                : {best_roc:.4f}")
print(f"\n  Saved outputs:")
print("    • outputs/task1_model_comparison.csv")
print("    • outputs/task1_model_comparison_bars.png")
print("    • outputs/task1_heatmap.png")
print("    • outputs/task1_confusion_matrix_best.png")
print("    • outputs/task1_roc_curves.png")
print("    • outputs/task1_cv_f1.png")
if SHAP_AVAILABLE:
    print("    • outputs/task1_shap_summary.png")
    print("    • outputs/task1_shap_bar.png")
print("=" * 90)